# LIBRAIRIES

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from affine import Affine
from scipy.ndimage import gaussian_filter
from rasterio.features import rasterize, geometry_mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as mpatches
import fiona
import rasterio
from rasterio.enums import MergeAlg
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point
import libpysal
import esda
import rasterstats
import warnings
warnings.filterwarnings("ignore")
import math
from statsmodels.nonparametric.smoothers_lowess import lowess
import matplotlib.cm as cm

import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from utils.config import *
from utils.functions import *

# PATH

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

# ─── Paths ──────────────────────────────────────────────────────────────────
input_file_path_PL = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/INPUT/"
output_file_path_PL = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/OUTPUT/"
input_file_path_PL_wave1 = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/WAVE1_MOBILITY/INPUT/"
output_file_path_PL_wave1 = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/WAVE1_MOBILITY/OUTPUT/"
output_file_path_PL_RASTER = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/OUTPUT/RASTER/"

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/GE/step-1/'
output_step2_path='../../Data/output/GE/step-2/'
output_step3_path='../../Data/output/GE/step-3/'

# DATA LOADING

In [ ]:
# ─── Chargement tous modes, filtres qualité + intra_GE ou intra_VD ───────────
layers     = fiona.listlayers(f'{input_file_path_PL}241120_legs.gpkg')
all_layers = []

for layer in layers:
    where_clause = """
        extreme99_length_mode = 0 AND
        extreme98_length_mode = 0 AND
        usr_w_constant_bad_signal = 0 AND
        (intra_GE = 1 OR intra_VD = 1)
    """

    gdf = gpd.read_file(
        f'{input_file_path_PL}241120_legs.gpkg',
        layer=layer,
        where=where_clause
    )
    print(f"── {layer:<25} selected: {len(gdf):>6}")
    if len(gdf) > 0:
        gdf["source_layer"] = layer
        all_layers.append(gdf)

if len(all_layers) == 0:
    print("⚠ Aucune couche chargée — vérifier le where_clause")
else:
    legs_filtered = gpd.GeoDataFrame(
        pd.concat(all_layers, ignore_index=True),
        crs="EPSG:4326"
    )
    del all_layers
    print(f"\nlegs_filtered : {len(legs_filtered)} legs / {legs_filtered['user_id_fors'].nunique()} users")
    print(f"  dont intra_GE : {(legs_filtered['intra_GE'] == 1).sum():,}")
    print(f"  dont intra_VD : {(legs_filtered['intra_VD'] == 1).sum():,}")

In [ ]:
# ─── Sauvegarde legs_filtered ─────────────────────────────────────────────────
legs_GE_VD_all = legs_filtered.copy()

In [ ]:
"""legs_GE_VD_all = gpd.read_parquet(f'{output_file_path_PL}legs_GE_VD_all.parquet')
"""

In [ ]:
user_stat  = pd.read_csv(f'{input_file_path_PL}241120_user_statistics.csv')

In [ ]:
communes_suisse = gpd.read_file(
    input_file_path_PL + "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg",
    layer="tlm_hoheitsgebiet"
)

In [ ]:
# ─── Chargement des cantons ───────────────────────────────────────────────────
cantons = gpd.read_file(
    f'{input_file_path_PL}swissBOUNDARIES3D_1_5_LV95_LN02.gpkg',
    layer='tlm_kantonsgebiet'
).to_crs(epsg=2056)

canton_VD = cantons[cantons["name"] == "Vaud"].copy()
canton_GE = cantons[cantons["name"] == "Genève"].copy()

print(f"Canton VD : {len(canton_VD)} polygone(s)")
print(f"Canton GE : {len(canton_GE)} polygone(s)")

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
fig.suptitle("Legs GE + VD — tous modes", fontsize=13, fontweight="bold")

legs_GE_VD_all.to_crs(epsg=2056).plot(
    ax=ax, color="#4C72B0", linewidth=0.3, alpha=0.3)

canton_GE.boundary.plot(ax=ax, color='red',   linewidth=1.5, label="Genève")
canton_VD.boundary.plot(ax=ax, color='green', linewidth=1.5, label="Vaud")

ax.legend(fontsize=9)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
both = legs_GE_VD_all[
    (legs_GE_VD_all["intra_GE"] == 1) & 
    (legs_GE_VD_all["intra_VD"] == 1)
]

print(f"Legs intra_GE = 1 ET intra_VD = 1 : {len(both):,}")
print(f"Users concernés                    : {both['user_id_fors'].nunique()}")

In [ ]:
print(legs_GE_VD_all.dtypes.to_string())

In [ ]:
# ─── Calcul n_days_GE et n_days_VD par user ───────────────────────────────────
n_days_GE = (
    legs_GE_VD_all[legs_GE_VD_all["intra_GE"] == 1]
    .groupby("user_id_fors")["legs_date"]
    .nunique()
    .rename("n_days_GE")
)

n_days_VD = (
    legs_GE_VD_all[legs_GE_VD_all["intra_VD"] == 1]
    .groupby("user_id_fors")["legs_date"]
    .nunique()
    .rename("n_days_VD")
)

# ─── Merge directement sur legs_GE_VD_all ────────────────────────────────────
legs_GE_VD_all = legs_GE_VD_all.merge(
    pd.DataFrame({"n_days_GE": n_days_GE, "n_days_VD": n_days_VD})
    .fillna(0).astype(int).reset_index(),
    on="user_id_fors",
    how="left"
)

print(f"Legs total     : {len(legs_GE_VD_all)}")
print(f"Colonnes ajout : n_days_GE, n_days_VD")
print(legs_GE_VD_all[["user_id_fors", "n_days_GE", "n_days_VD"]].drop_duplicates().head(5))

In [ ]:
# ─── Calcul n_days_GE_VD par user (dates uniques tous cantons confondus) ──────
n_days_GE_VD = (
    legs_GE_VD_all
    .groupby("user_id_fors")["legs_date"]
    .nunique()
    .rename("n_days_GE_VD")
)

# ─── Merge directement sur legs_GE_VD_all ────────────────────────────────────
legs_GE_VD_all = legs_GE_VD_all.merge(
    n_days_GE_VD.reset_index(),
    on="user_id_fors",
    how="left"
)

print(f"Legs total      : {len(legs_GE_VD_all)}")
print(f"Colonne ajoutée : n_days_GE_VD")
print(f"\nStats n_days_GE_VD :")
print(legs_GE_VD_all[["user_id_fors", "n_days_GE_VD"]]
      .drop_duplicates()["n_days_GE_VD"]
      .describe().round(1))

In [ ]:
print(legs_GE_VD_all.columns.tolist())

In [ ]:
legs_GE_VD_all

In [ ]:
# ─── Filtrer uniquement les déplacements à pied ───────────────────────────────
legs_GE_VD_walk = legs_GE_VD_all[
    legs_GE_VD_all["mode"] == "Mode::Walk"
].copy()

print(f"Legs total avant filtre : {len(legs_GE_VD_all):,}")
print(f"Legs Mode::Walk         : {len(legs_GE_VD_walk):,}")
print(f"Users uniques           : {legs_GE_VD_walk['user_id_fors'].nunique()}")

In [ ]:
# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
fig.suptitle("Legs GE + VD — Walk", fontsize=13, fontweight="bold")

legs_GE_VD_walk.to_crs(epsg=2056).plot(
    ax=ax, color="#4C72B0", linewidth=0.3, alpha=0.3)

canton_GE.boundary.plot(ax=ax, color='red',   linewidth=1.5, label="Genève")
canton_VD.boundary.plot(ax=ax, color='green', linewidth=1.5, label="Vaud")

ax.legend(fontsize=9)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# ─── Chargement de la grille 500m ─────────────────────────────────────────────
grid_500 = gpd.read_file(
    f'{input_file_path_PL}500_grid_Vaud_Geneva_within_no_lake/500_grid_Vaud_Geneva_within_no_lake.shp'
).to_crs(epsg=2056)

print(f"Grid 500m total : {len(grid_500)}")
print(f"CRS             : {grid_500.crs}")
print(f"Colonnes        : {grid_500.columns.tolist()}")
print(f"Bounds          : {grid_500.total_bounds}")

In [ ]:
n_days_per_user = legs_GE_VD_walk.groupby("user_id_fors")["n_days_GE_VD"].first().to_dict()

In [ ]:
n_days_per_user

In [ ]:
resolution = int(round(np.sqrt(grid_500.geometry.iloc[0].area)))

xmin, ymin, xmax, ymax = grid_500.total_bounds

raster_width  = int(round((xmax - xmin) / resolution))
raster_height = int(round((ymax - ymin) / resolution))

transform = Affine(resolution, 0, xmin,
                   0, -resolution, ymax)
canton_mask = np.ones((raster_height, raster_width), dtype=bool)

In [ ]:
from rasterio.features import rasterize as rio_rasterize

canton_mask = rio_rasterize(
    shapes=((geom, 1) for geom in grid_500.geometry),
    out_shape=(raster_height, raster_width),
    transform=transform,
    fill=0,
    dtype="uint8"
).astype(bool)

print(f"pixels actifs dans le masque : {canton_mask.sum()}")
# doit être proche de 13243

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Grille donnée
grid_500.plot(ax=axes[0], edgecolor="grey", facecolor="none")
axes[0].set_title("grid_500")

# Raster canton_mask
axes[1].imshow(canton_mask, origin="upper", cmap="Greys")
axes[1].set_title("canton_mask")

plt.tight_layout()
plt.show()

In [ ]:
print(f"raster_width  : {raster_width}")
print(f"raster_height : {raster_height}")
print(f"n cellules grid_500 : {len(grid_500)}")
print(f"raster total cellules : {raster_width * raster_height}")

print(f"\nbounds grid_500 : {grid_500.total_bounds}")
print(f"xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}")
print(f"transform : {transform}")

In [ ]:
legs_GE_VD_walk = legs_GE_VD_walk.to_crs(2056)

In [ ]:
raster, meta = build_density_raster_v2(
    gdf_group       = legs_GE_VD_walk,
    raster_height   = raster_height,
    raster_width    = raster_width,
    transform       = transform,
    canton_mask     = canton_mask,
    n_days_per_user = n_days_per_user,
    sigma           = 1,
    verbose         = True,
    output_path     = output_file_path_PL_RASTER,
    raster_name     = "density_GE_VD"

)

In [ ]:
print(meta)

In [ ]:
print(legs_GE_VD_walk.crs)
print(f"transform origin : {xmin}, {ymax}")
print(legs_GE_VD_walk.total_bounds)

In [ ]:
# ─── Normalisation ────────────────────────────────────────────────────────────
d_max       = compute_scale([raster], clip_percentile=95, mode=norm_mode)
raster_norm = normalize_raster(raster, d_max, mode=norm_mode)

In [ ]:
communes_clip = communes_suisse.clip(grid_500.total_bounds)

# ─── Plot ─────────────────────────────────────────────────────────────────────
plot_density_map(
    raster_norm,
    title=f"Mean Daily Pedestrian Density — All users\nn_users={meta['n_users_valid']} | norm={norm_mode} | P{95}",
    extent=grid_500.total_bounds[[0, 2, 1, 3]],
    canton_GE=None,
    girec=communes_clip,
    clip_percentile_label=95,
)

# EXPORTS

In [ ]:
# ─── Parquet All──────────────────────────────────────────────────────────────────
legs_GE_VD_all.to_parquet(f'{output_file_path_PL}legs_GE_VD_all.parquet')
print(f"✓ legs_GE_VD_all.parquet sauvegardé : {len(legs_GE_VD_all)} legs")


# ─── GPKG All─────────────────────────────────────────────────────────────────────
legs_GE_VD_all.to_file(
    f'{output_file_path_PL}legs_GE_VD_all.gpkg',
    layer='legs_GE_VD_all',
    driver='GPKG'
)
print(f"✓ legs_GE_VD_all.gpkg sauvegardé")

In [ ]:
# ─── Parquet Walk──────────────────────────────────────────────────────────────────
legs_GE_VD_walk.to_parquet(f'{output_file_path_PL}legs_GE_VD_walk.parquet')
print(f"✓ legs_GE_VD_walk.parquet sauvegardé : {len(legs_GE_VD_all)} legs")


# ─── GPKG Walk─────────────────────────────────────────────────────────────────────
legs_GE_VD_walk.to_file(
    f'{output_file_path_PL}legs_GE_VD_walk.gpkg',
    layer='legs_GE_VD_walk',
    driver='GPKG'
)
print(f"✓ legs_GE_VD_walk.gpkg sauvegardé")